# CNN Image Classification Benchmark

## 1. Project Overview
This notebook runs controlled CNN experiments for two tasks: smile binary classification and SIGNS multiclass digit classification. It compares baseline, improved, and augmented models, then summarizes metrics and plots produced by the training pipeline.

## 2. Environment Setup

In [ ]:
from pathlib import Path
import pandas as pd

from src import config
from src.data_loader import MissingDatasetError, load_happy_dataset, load_signs_dataset, dataset_summary, print_dataset_summary
from src.train import available_models, train_and_evaluate
from src.visualization import plot_benchmark_comparison

config.ensure_output_dirs()
config.set_global_determinism(config.SEED)
print(f"Project root: {config.PROJECT_ROOT}")
print(f"Outputs: {config.OUTPUT_DIR}")

## 3. Dataset Loading and Inspection

In [ ]:
datasets = {}
try:
    datasets["smile"] = load_happy_dataset()
    datasets["signs"] = load_signs_dataset()
except MissingDatasetError as exc:
    print(str(exc))
    raise SystemExit(1)

print_dataset_summary(datasets["smile"], "smile")
print_dataset_summary(datasets["signs"], "signs")

summary_df = pd.DataFrame([
    dataset_summary(datasets["smile"], "smile"),
    dataset_summary(datasets["signs"], "signs"),
])
summary_df

## 4. Baseline CNN Models
- Smile: `build_smile_baseline()`
- SIGNS: `build_signs_baseline()`

## 5. Improved CNN Models
- Smile: `build_smile_improved_cnn()` and `build_smile_augmented_cnn()`
- SIGNS: `build_signs_improved_cnn()` and `build_signs_augmented_cnn()`

## 6. Training and Evaluation

In [ ]:
benchmark_plan = {
    "smile": ["baseline", "improved_cnn", "augmented_cnn"],
    "signs": ["baseline", "improved_cnn", "augmented_cnn"],
}

results = []
for task, model_names in benchmark_plan.items():
    data = datasets[task]
    models = available_models(task)
    task_epochs = config.SMILE_EPOCHS if task == "smile" else config.SIGNS_EPOCHS
    batch_size = config.SMILE_BATCH_SIZE if task == "smile" else config.DEFAULT_BATCH_SIZE

    for model_name in model_names:
        print(f"\nRunning {task}/{model_name}")
        _, _, summary = train_and_evaluate(
            task=task,
            model_name=model_name,
            model_builder=models[model_name],
            data=data,
            epochs=task_epochs,
            batch_size=batch_size,
            seed=config.SEED,
        )
        summary["seed"] = config.SEED
        results.append(summary)

results_df = pd.DataFrame(results)
results_df

## 7. Benchmark Results

In [ ]:
results_df = results_df.sort_values(["task", "model", "seed"]).reset_index(drop=True)
results_csv = config.METRICS_DIR / "benchmark_results.csv"
results_json = config.METRICS_DIR / "benchmark_results.json"

results_df.to_csv(results_csv, index=False)
results_df.to_json(results_json, orient="records", indent=2)

print(f"Saved: {results_csv}")
print(f"Saved: {results_json}")
results_df

### README-Ready Results Export
Generate README-style markdown tables and a short discussion directly from `outputs/metrics/benchmark_results.csv`.
This cell also writes a reusable snippet to `outputs/metrics/readme_results_snippet.md`.

In [ ]:
from pathlib import Path
import pandas as pd

benchmark_csv = config.METRICS_DIR / "benchmark_results.csv"
if "results_df" in globals() and isinstance(results_df, pd.DataFrame) and not results_df.empty:
    df = results_df.copy()
elif benchmark_csv.exists():
    df = pd.read_csv(benchmark_csv)
else:
    raise FileNotFoundError(f"Could not find benchmark results at {benchmark_csv}")

df = df.sort_values(["task", "test_accuracy"], ascending=[True, False]).reset_index(drop=True)


def _fnum(value, digits=4):
    if pd.isna(value):
        return "-"
    return f"{float(value):.{digits}f}"


def _table_for_task(frame: pd.DataFrame, task: str) -> str:
    task_df = frame[frame["task"] == task].copy()
    if task_df.empty:
        return f"### {task.capitalize()} task\nNo results available.\n"

    if task == "smile":
        columns = [
            ("model", "Model"),
            ("epochs_trained", "Epochs trained"),
            ("best_validation_accuracy", "Best val accuracy"),
            ("test_accuracy", "Test accuracy"),
            ("test_loss", "Test loss"),
            ("roc_auc", "ROC-AUC"),
            ("training_time_seconds", "Training time (s)"),
        ]
    else:
        columns = [
            ("model", "Model"),
            ("epochs_trained", "Epochs trained"),
            ("best_validation_accuracy", "Best val accuracy"),
            ("test_accuracy", "Test accuracy"),
            ("test_loss", "Test loss"),
            ("training_time_seconds", "Training time (s)"),
        ]

    header = "| " + " | ".join(label for _, label in columns) + " |"
    sep = "|" + "|".join("---" for _ in columns) + "|"
    lines = [f"### {task.capitalize()} task", header, sep]

    for _, row in task_df.iterrows():
        vals = []
        for key, _ in columns:
            value = row.get(key)
            if key == "model":
                vals.append(f"`{value}`")
            elif key in {"epochs_trained"}:
                vals.append(str(int(value)) if pd.notna(value) else "-")
            elif key == "training_time_seconds":
                vals.append(_fnum(value, digits=2))
            else:
                vals.append(_fnum(value, digits=4))
        lines.append("| " + " | ".join(vals) + " |")

    return "\n".join(lines) + "\n"


def _quick_discussion(frame: pd.DataFrame) -> str:
    lines = ["### Quick discussion"]

    for task in sorted(frame["task"].unique()):
        task_df = frame[frame["task"] == task].copy().sort_values("test_accuracy", ascending=False)
        if task_df.empty:
            continue
        top = task_df.iloc[0]
        lines.append(
            f"- {task.capitalize()}: best test accuracy is `{_fnum(top['test_accuracy'])}` with `{top['model']}`."
        )

        baseline = task_df[task_df["model"] == "baseline"]
        if not baseline.empty and top["model"] != "baseline":
            b = baseline.iloc[0]
            delta = float(top["test_accuracy"] - b["test_accuracy"])
            lines.append(
                f"- {task.capitalize()}: `{top['model']}` vs baseline delta is `{delta:+.4f}` test-accuracy points."
            )

    seeds = sorted(frame.get("seed", pd.Series(dtype=int)).dropna().astype(int).unique().tolist()) if "seed" in frame.columns else []
    if seeds:
        lines.append(f"- Results shown from seed set: `{seeds}`.")

    return "\n".join(lines) + "\n"

seed_info = "unknown"
if "seed" in df.columns and df["seed"].notna().any():
    seeds = sorted(df["seed"].dropna().astype(int).unique().tolist())
    seed_info = str(seeds)

sections = [
    f"Latest local benchmark snapshot (seed set `{seed_info}`, generated from `outputs/metrics/benchmark_results.csv`):\n",
    _table_for_task(df, "smile"),
    _table_for_task(df, "signs"),
    _quick_discussion(df),
    "### Figures produced by the run\n"
    "- `outputs/figures/benchmark_test_accuracy.png`\n"
    "- `outputs/figures/smile_*_training_curves.png`, `outputs/figures/smile_*_confusion_matrix.png`\n"
    "- `outputs/figures/signs_*_training_curves.png`, `outputs/figures/signs_*_confusion_matrix.png`\n",
]

readme_results_md = "\n".join(sections)

snippet_path = config.METRICS_DIR / "readme_results_snippet.md"
snippet_path.write_text(readme_results_md, encoding="utf-8")

print(f"Saved README snippet: {snippet_path}")
print("\n" + readme_results_md)


## 8. Visual Analysis

In [ ]:
plot_benchmark_comparison(results_df, output_path=config.FIGURES_DIR / "benchmark_test_accuracy.png")
print(f"Saved figure: {config.FIGURES_DIR / 'benchmark_test_accuracy.png'}")

## 9. Summary of Findings
This notebook executes a reproducible benchmark across baseline, improved, and augmented CNN variants for both tasks. Detailed per-run training logs, evaluation metrics, confusion matrices, and benchmark summary artifacts are saved under `outputs/metrics/` and `outputs/figures/`.